<a href="https://colab.research.google.com/github/kerenG99/UFZ-Helmoltz-Summer-School-2026/blob/main/notebooks/03_data_structures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 · Data structures

> **Trends in multi-omics data analysis for Microbial Ecology and Biotechnology** - Summer School, UFZ Leipzig  
> *Refreshing / Basic Knowledge: Python* - Instructor: Anderson Santos

## Learning objectives
- store collections in **lists**, **tuples**, **sets** and **dictionaries**;
- use list **methods**, **slicing** and **comprehensions**;
- avoid the **aliasing** bug when copying lists;
- use **sets** to compare taxa between samples;
- use **dictionaries** for id→sequence and codon→amino-acid maps.

> **How to read this notebook while teaching.** Every code cell is commented: the lines starting with `#` explain what the code does and why it matters biologically, so you can narrate the class straight from the cell even without notes. Blockquotes marked **Instructor note** are extra talking points.

---
| Notebook | Topic |
|---|---|
| 01 | Python essentials & operators |
| 02 | Strings & biological sequences |
| 03 | Data structures |
| 04 | Control flow & functions |
| 05 | Files, scripting & modules |
| 06 | Pandas for omics |

## Setup — run this cell first

This cell makes the example data available whether you are on **your own computer** or on **Google Colab**. You do not need to understand it. Click it and press **Shift+Enter**.

> **Instructor note.** Set `GITHUB_REPO` below to your repository once, before sharing the notebooks.

In [1]:
# Run me first. Works on a local install AND on Google Colab.
import os

GITHUB_REPO = "andersonavilasantos/ufz-summerschool-python"   # <-- set to your GitHub repo

if not os.path.exists("../data/asv_table.csv"):
    # Data not found locally -> assume Google Colab and download the course files.
    os.system(f"git clone -q https://github.com/{GITHUB_REPO}.git course_repo")
    if os.path.exists("course_repo/notebooks"):
        os.chdir("course_repo/notebooks")

assert os.path.exists("../data/asv_table.csv"), (
    "Could not find the data. On Colab, check GITHUB_REPO above; "
    "locally, run the notebook from inside the notebooks folder.")
print("✅ Setup complete — the data folder is available.")

✅ Setup complete — the data folder is available.


## 1. Lists — ordered, mutable collections
Think: the taxa observed in a sample.

In [2]:
# A list holds items IN ORDER, inside square brackets [ ].
# Here: the taxa detected in one sample.
taxa = ['Pseudomonas', 'Bacteroides', 'Nitrospira', 'Bacillus']
print('count   :', len(taxa))   # how many items
print('first   :', taxa[0])     # indexing works like strings: 0-based
print('last    :', taxa[-1])    # -1 = last item
print('slice   :', taxa[1:3])   # a sub-list (items 1 and 2)

count   : 4
first   : Pseudomonas
last    : Bacillus
slice   : ['Bacteroides', 'Nitrospira']


Lists can be **nested** (a list of samples, each a list of taxa):

In [5]:
# A list can contain other lists -> a 2D-ish structure.
samples = [['Pseudomonas', 'Bacillus'], ['Nitrospira', 'Bacteroides']]
print(samples[0][1])   # first sample -> its second taxon = 'Bacillus'
print(samples[1][0])

Bacillus
Nitrospira


In [ ]:
#have taxa, then divdied them into samples

### Built-in functions on lists

In [6]:
# Python has ready-made functions that summarise a list of numbers.
counts = [120, 305, 88, 412, 15]
print('n     =', len(counts))              # number of values
print('min   =', min(counts), ' max =', max(counts))
print('total =', sum(counts))              # add them all up
print('Nitrospira' in taxa)                # membership test, like strings
print('longest name:', max(taxa, key=len)) # 'key=len' -> compare BY length

n     = 5
min   = 15  max = 412
total = 940
True
longest name: Pseudomonas


### List methods — grow, shrink, reorder

In [9]:
# Methods that ADD items:
lst = [120, 305, 88]
lst.append(412)            # add ONE item to the end
lst.extend([15, 60])       # add SEVERAL items (note: not append!)
lst.insert(0, 999)         # insert at a position (here, the front)
lst.insert(-1, 68)
print('after adds :', lst)
print('index of 88:', lst.index(88))   # WHERE is this value?
print("position of 68:", lst.index(68))
print('count of 15:', lst.count(15))   # HOW MANY of this value?

after adds : [999, 120, 305, 88, 412, 15, 68, 60]
index of 88: 3
position of 68: 6
count of 15: 1


In [10]:
# Methods that REMOVE items:
lst.pop()                  # remove & return the LAST item
lst.pop(0)                 # remove & return item at a position
lst.remove(88)             # remove the first matching VALUE
print('after removes:', lst)

# Sorting:
lst.sort()                 # ascending sort IN PLACE (changes lst itself)
print('sorted     :', lst)
lst.sort(reverse=True)     # descending
print('descending :', lst)
print('sorted() copy:', sorted([3, 1, 2]))  # sorted() returns a NEW list, leaves original

after removes: [120, 305, 412, 15, 68]
sorted     : [15, 68, 120, 305, 412]
descending : [412, 305, 120, 68, 15]
sorted() copy: [1, 2, 3]


## 2. Copying a list — the aliasing trap
Assigning a list does **not** copy it: both names point to the *same* list. This catches most people once. Slice `[:]` (or `list(...)`) to make a real copy.

In [13]:
# The classic beginner bug. '=' does NOT copy a list.
a = [3, 1, 2]
b = a               # b is just another NAME for the SAME list
b.append(99)
print('a =', a, '  <- a changed too, because a and b are one list!')
print(a,b)


a = [3, 1, 2, 99]   <- a changed too, because a and b are one list!
[3, 1, 2, 99] [3, 1, 2, 99]


In [11]:

# The fix: make a real copy with a full slice [:] (or list(a)).
a = [3, 1, 2]
b = a[:]            # an independent copy
b.append(99)
print('a =', a, '  b =', b, '  <- now they are independent')

a = [3, 1, 2, 99]   <- a changed too, because a and b are one list!
a = [3, 1, 2]   b = [3, 1, 2, 99]   <- now they are independent


> **Instructor note.** Worth dwelling on. This bug silently corrupts real analyses, for example editing a "copy" of a sample list that was never actually copied.

## 3. List comprehensions
A compact way to build a list from another. This is a common and idiomatic Python pattern.

In [15]:
# A comprehension builds a new list in one line: [ expression for item in iterable ]
seqs = ['ATGC', 'GGGCGC', 'ATAT', 'GCGCGCGC']

# 1) transform every item -> here, its length:
lengths = [len(s) for s in seqs]
print('lengths:', lengths)



lengths: [4, 6, 4, 8]


In [16]:
# 2) compute GC for each sequence:
gc = [(s.count('G') + s.count('C')) / len(s) for s in seqs]
print('GC     :', [round(x, 2) for x in gc])



GC     : [0.5, 1.0, 0.0, 1.0]


In [17]:
# 3) add an 'if' to FILTER - keep only GC-rich sequences:
high_gc = [s for s in seqs if (s.count('G') + s.count('C')) / len(s) > 0.5]
print('high-GC seqs:', high_gc)

high-GC seqs: ['GGGCGC', 'GCGCGCGC']


## 4. Tuples — immutable records
Good for fixed groupings, e.g. an amplicon region `(start, end)` or an (x, y) coordinate. They support **unpacking**.

In [18]:
# A tuple is like a list but IMMUTABLE - it cannot be changed after creation.
# Use it for fixed pairs/records: here, the V4 amplicon coordinates.
v4_region = (515, 806)
start, end = v4_region        # 'unpacking': give each element a name
print('amplicon length:', end - start)
# v4_region[0] = 500          # <- would raise TypeError (tuples are read-only)

amplicon length: 291


In [25]:
#dont get this tbh :(
v5_region = (700, 806)
start, end = v5_region
print('amplicon length:', end - start)

amplicon length: 106


## 5. Sets — unique, unordered
Perfect for 'which taxa are present?' and for comparing two samples.

In [26]:
# A set stores UNIQUE items, with no order. Curly braces { }.
# Ideal for community comparisons between two samples:
sample_A = {'Pseudomonas', 'Bacillus', 'Nitrospira', 'Bacteroides'}
sample_B = {'Bacillus', 'Nitrospira', 'Methanosarcina'}
print('shared (intersection):', sample_A & sample_B)  # in BOTH
print('all taxa (union)     :', sample_A | sample_B)  # in EITHER
print('only in A (diff)     :', sample_A - sample_B)  # in A but not B
print('exclusive (sym diff) :', sample_A ^ sample_B)  # in exactly ONE
print('is B subset of A?    :', sample_B <= sample_A)

shared (intersection): {'Bacillus', 'Nitrospira'}
all taxa (union)     : {'Methanosarcina', 'Bacteroides', 'Bacillus', 'Pseudomonas', 'Nitrospira'}
only in A (diff)     : {'Pseudomonas', 'Bacteroides'}
exclusive (sym diff) : {'Methanosarcina', 'Bacteroides', 'Pseudomonas'}
is B subset of A?    : False


In [35]:
sample_1 = {'A, B, C, D, E, F,'}
sample_2 = {'B, D, C, D, A, '}
  #What quaifqualifiesies it as a subsetubset?
print('how many letters:', set(sample_2))


how many letters: {'B, D, C, D, A, '}


In [29]:
# A very common use: de-duplicate a list by converting it to a set.
reads = ['Bacillus', 'Bacillus', 'Nitrospira', 'Bacillus', 'Nitrospira']
print('unique taxa:', set(reads))

unique taxa: {'Bacillus', 'Nitrospira'}


## 6. Dictionaries — key → value maps
A natural structure for biological data: id→sequence, codon→amino acid, sample→measurement.

In [37]:
# A dictionary maps a KEY to a VALUE: {key: value, ...}.
# zip() pairs up two lists element-by-element; dict() turns pairs into a map.
ids  = ['ASV_1', 'ASV_2', 'ASV_3']
seqs = ['ATGGGC', 'GCGCGCGC', 'ATATAT']
records = dict(zip(ids, seqs))   # {'ASV_1': 'ATGGGC', ...}
print(records)
print('ASV_2 ->', records['ASV_2'])   # look up a value BY its key

{'ASV_1': 'ATGGGC', 'ASV_2': 'GCGCGCGC', 'ASV_3': 'ATATAT'}
ASV_2 -> GCGCGCGC


In [ ]:
# The genetic code is naturally a dictionary: codon -> amino acid.
# We build all 64 entries with a comprehension (a classic Python trick).
bases = 'TCAG'
aa = 'FFLLSSSSYY**CC*WLLLLPPPPHHQQRRRRIIIMTTTTNNKKSSRRVVVVAAAADDEEGGGG'
codon_table = {a + b + d: aa[i]
               for i, (a, b, d) in enumerate(
                   (x, y, z) for x in bases for y in bases for z in bases)}
print('ATG codes for', codon_table['ATG'])   # M = Met = start codon
print('TAA codes for', codon_table['TAA'])   # * = stop
print('number of codons defined:', len(codon_table))   # 64

Dictionary essentials — `keys`, `values`, `items`, `get`, membership:

In [38]:
abund = {'Pseudomonas': 305, 'Bacillus': 120, 'Nitrospira': 88}
print('taxa   :', list(abund.keys()))     # just the keys
print('counts :', list(abund.values()))   # just the values


# .items() gives key,value pairs - the usual way to loop over a dict:
for taxon, n in abund.items():
    print(f'  {taxon:12} {n}')

# .get() returns a default instead of crashing when a key is missing:
print('missing taxon ->', abund.get('Geobacter', 0))   # 0 instead of an error

taxa   : ['Pseudomonas', 'Bacillus', 'Nitrospira']
counts : [305, 120, 88]
  Pseudomonas  305
  Bacillus     120
  Nitrospira   88
missing taxon -> 0


In [45]:
abund.get("bas" , 0) #why is it that adding a zero at the end geenrates a zero
abund.get("Bacillus")


120

---
## Exercises
**E1.** From `seqs = ['ATGC','GGGG','ATATAT','GCGC']`, build a list of `(sequence, length)` **tuples** with a comprehension.

**E2.** Two samples: `A = {'Pseudomonas','Bacillus','Nitrospira'}` and `B = {'Bacillus','Geobacter'}`. Print the taxa **unique to A**.

**E3.** Given `abund = {'Pseudomonas':305,'Bacillus':120,'Nitrospira':88}`, use a comprehension to build a dict of **relative abundances** (each count divided by the total).

**E4.** Using `codon_table`, translate the codons `['ATG','GGC','TTT','TAA']` into a list of amino acids.

In [ ]:
# your code here

<details>
<summary><b>Solution: E1-E4</b> (click to expand)</summary>

```python
seqs = ['ATGC','GGGG','ATATAT','GCGC']
print([(s, len(s)) for s in seqs])
A = {'Pseudomonas','Bacillus','Nitrospira'}; B = {'Bacillus','Geobacter'}
print(A - B)
abund = {'Pseudomonas':305,'Bacillus':120,'Nitrospira':88}
total = sum(abund.values())
print({k: round(v/total, 3) for k, v in abund.items()})
print([codon_table[c] for c in ['ATG','GGC','TTT','TAA']])
```
</details>

### Recap
- **list** (ordered, mutable) with rich methods; slice `[:]` to copy safely.
- **comprehensions** build lists/dicts concisely, with optional `if`.
- **tuple** (immutable, unpackable); **set** (unique) for comparing communities.
- **dict** maps keys→values: `zip`, comprehension, `items()`, `get()`.

Next: **04 · Control flow & functions**.